### LAMA (Lights Out) – Lösung Schritt für Schritt

*35. Bundeswettbewerb Informatik (2016), 1. Runde, Junioraufgabe 2*

Ein neues „Gadget“ ist auf dem Markt: das LAMA (LED Advanced Matrix Arrangement). Es handelt sich um ein Schaltbrett, auf
dem 25 mit je einer LED versehene Tasten in einem Raster mit 5 Reihen und 5 Spalten angeordnet sind. Die LED einer Taste
kann entweder an- oder ausgeschaltet sein. Wenn eine Taste gedrückt wird, führt das LAMA eine mit der Taste verbundene
Aktion aus. Die Aktionen sind programmierbar, so dass man mit dem LAMA verschiedene Spiele umsetzen kann.

Xaver wünscht sich so ein LAMA, um „Lights Out“ darauf programmieren und spielen zu können. Bei diesem Spiel sind die LEDs
einiger Tasten zu Beginn eingeschaltet, und es geht darum, alle LEDs auszuschalten. Dazu darf man in jedem Spielzug eine
Taste drücken. Solch ein Spielzug schaltet die LED der gedrückten Taste und ihre (bis zu vier) Nachbar-LEDs um.
Sind alle LEDs ausgeschaltet, ist das Spiel beendet.

Leider ist es noch eine Weile bis Weihnachten. In der Zwischenzeit programmiert Xavers Schwester Yanelle mit ihm Lights Out
„ganz normal“ am Computer.

#### Junioraufgabe 2

1. Schreibe ein Programm, mit dem man Lights Out wie auf dem LAMA spielen kann. Beim Start des Spiels muss zuerst
   festgelegt werden, welche LEDs zu Beginn eingeschaltet sind. Dann soll gespielt werden können; der Spieler soll also
   seine Züge eingeben können. Zu Beginn und nach jedem Spielzug soll der Zustand der LEDs angezeigt werden.
2. Die Hersteller denken darüber nach, LAMAs auch in anderen Größen zu produzieren. Erweitere dein Programm so, dass man
   auch Tasten-Raster mit anderen Anzahlen von Reihen und Spalten (bis zu einer von dir bestimmten Maximalzahl) bespielen kann.
3. Erweitere dein Programm so, dass auch die folgende Variante von Lights Out gespielt werden kann: Hier schaltet ein
   Spielzug die LED der gedrückten Taste und alle LEDs in dem links und oberhalb der Taste gelegenen Tasten-Rechteck um.
   Auch bei dieser Variante ist das Spiel beendet, wenn alle LEDs ausgeschaltet sind.

Beispieldaten gibt es zu dieser Aufgabe nicht (interaktives Spiel).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zustand und Zustandsänderung:** Das Spielfeld ist ein Raster aus 0 (LED aus) und 1 (LED an), also eine Liste von Listen. Ein Spielzug ändert den Zustand. Das Umschalten geht mit `1 - wert`.
- **Nachbarn mit Randprüfung:** Eine Taste am Rand hat weniger als vier Nachbarn. Man prüft für jeden Nachbarn, ob er im Raster liegt.
- **Regeln als Daten:** Beide Spielvarianten unterscheiden sich nur darin, **welche Felder** ein Zug umschaltet. Wir schreiben eine Funktion, die diese Felder liefert (abhängig von der gewählten Regel),
  und benutzen sie überall. So lässt sich eine neue Variante mit wenigen Zeilen ergänzen.
- **Einfache Benutzereingabe:** `input(...)` liest eine Eingabe, `split()` und `int()` machen Zahlen daraus. Ungültige Eingaben fangen wir mit `try/except` ab.
- **Parameter statt Konstanten:** Reihen- und Spaltenzahl stehen nicht fest im Programm, sondern werden übergeben, dann funktioniert jede Größe.
- **Für Fortgeschrittene:** Das Spiel lässt sich **lösen**. Jede Taste muss höchstens einmal gedrückt werden, und die Reihenfolge ist egal. Mit „Lichter jagen“ (Chasing Lights) findet man die Lösung schnell,
  im Hintergrund steckt Rechnen modulo 2 (Gauß-Verfahren über dem Körper mit zwei Elementen).

**Nützliche Funktionen** (Bibliotheken braucht man keine, nur `random` für Zufallsstellungen)
- `input`, `split`, `int`, `try/except ValueError`, `all`, `any`, `random.randrange`, `list(zeile)` (Kopie einer Liste)

## Die Idee

**Das Brett.** `brett[z][s]` ist 1, wenn die LED in Reihe `z` und Spalte `s` an ist, sonst 0. Intern zählen wir ab 0, bei der Anzeige und bei den Eingaben ab 1 (wie es ein Mensch erwartet).

**Ein Zug** schaltet eine Liste von Feldern um. Welche Felder das sind, hängt von der Regel ab:
- **Regel `nachbarn`** (Lights Out): die gedrückte Taste und ihre Nachbarn oben, unten, links und rechts, soweit sie im Raster liegen.
- **Regel `obenlinks`** (die Variante): alle Tasten im Rechteck von der linken oberen Ecke bis zur gedrückten Taste.

**Das Spiel** ist vorbei, wenn keine LED mehr an ist.

## Schritt 1: Das Brett erzeugen und anzeigen

`neues_brett(reihen, spalten, an)` legt ein Brett mit lauter Nullen an. Die Liste `an` enthält die Positionen der eingeschalteten LEDs als Paare `(Reihe, Spalte)` (ab 1 gezählt).
Wir bauen die Zeilen mit einer Schleife, damit jede Zeile eine **eigene** Liste ist (mit `[[0] * spalten] * reihen` wären alle Zeilen dieselbe Liste).

`zeige(brett)` gibt das Brett aus: `#` für eine eingeschaltete LED, `.` für eine ausgeschaltete. Oben und links stehen die Nummern der Spalten und Reihen.

In [ ]:
MAX_GROESSE = 20                            # größtes erlaubtes Raster: 20 x 20


def neues_brett(reihen, spalten, an):
    brett = []
    for z in range(reihen):
        zeile = []
        for s in range(spalten):
            zeile.append(0)
        brett.append(zeile)
    for z, s in an:
        brett[z - 1][s - 1] = 1             # Eingabe ab 1, intern ab 0
    return brett


def zeige(brett):
    kopf = '   '
    for s in range(len(brett[0])):
        kopf = kopf + f'{s + 1:2} '
    print(kopf)
    for z in range(len(brett)):
        text = f'{z + 1:2} '
        for s in range(len(brett[0])):
            text = text + (' # ' if brett[z][s] == 1 else ' . ')
        print(text)


brett = neues_brett(5, 5, [(2, 3), (3, 2), (3, 3), (3, 4), (4, 3)])
zeige(brett)

    1  2  3  4  5 
 1  .  .  .  .  . 
 2  .  .  #  .  . 
 3  .  #  #  #  . 
 4  .  .  #  .  . 
 5  .  .  .  .  . 


## Schritt 2: Welche Felder schaltet ein Zug um?

`felder_des_zugs(z, s, reihen, spalten, regel)` gibt die Liste der Felder zurück, die der Druck auf die Taste `(z, s)` (ab 0 gezählt) umschaltet.

- Bei `nachbarn` fangen wir mit der Taste selbst an und fügen jeden der vier Nachbarn hinzu, wenn er im Raster liegt (`0 <= zeile < reihen` und `0 <= spalte < spalten`).
- Bei `obenlinks` laufen zwei Schleifen über alle Reihen von 0 bis `z` und alle Spalten von 0 bis `s`.

In [ ]:
def felder_des_zugs(z, s, reihen, spalten, regel):
    felder = []
    if regel == 'nachbarn':
        felder.append((z, s))
        for dz, ds in [(-1, 0), (1, 0), (0, -1), (0, 1)]:       # oben, unten, links, rechts
            nz = z + dz
            ns = s + ds
            if 0 <= nz < reihen and 0 <= ns < spalten:
                felder.append((nz, ns))
    else:                                                       # Regel 'obenlinks'
        for nz in range(z + 1):
            for ns in range(s + 1):
                felder.append((nz, ns))
    return felder


print('Ecke (0, 0):  ', felder_des_zugs(0, 0, 5, 5, 'nachbarn'))
print('Mitte (2, 2): ', felder_des_zugs(2, 2, 5, 5, 'nachbarn'))
print('Variante (1, 2):', felder_des_zugs(1, 2, 5, 5, 'obenlinks'))

Ecke (0, 0):   [(0, 0), (1, 0), (0, 1)]
Mitte (2, 2):  [(2, 2), (1, 2), (3, 2), (2, 1), (2, 3)]
Variante (1, 2): [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]


## Schritt 3: Einen Zug ausführen und das Ende erkennen

`druecke(brett, z, s, regel)` schaltet alle Felder des Zugs um (`1 - wert` macht aus 0 eine 1 und aus 1 eine 0). `gewonnen(brett)` ist wahr, wenn in **allen** Zeilen **alle** Werte 0 sind.

In [ ]:
def druecke(brett, z, s, regel):
    for nz, ns in felder_des_zugs(z, s, len(brett), len(brett[0]), regel):
        brett[nz][ns] = 1 - brett[nz][ns]


def gewonnen(brett):
    for zeile in brett:
        if 1 in zeile:
            return False
    return True


brett = neues_brett(5, 5, [(2, 3), (3, 2), (3, 3), (3, 4), (4, 3)])
print('Start:')
zeige(brett)
druecke(brett, 2, 2, 'nachbarn')            # Taste in Reihe 3, Spalte 3 (ab 0 gezählt: 2, 2)
print('Nach dem Druck auf Reihe 3, Spalte 3:')
zeige(brett)
print('Gewonnen:', gewonnen(brett))

Start:
    1  2  3  4  5 
 1  .  .  .  .  . 
 2  .  .  #  .  . 
 3  .  #  #  #  . 
 4  .  .  #  .  . 
 5  .  .  .  .  . 
Nach dem Druck auf Reihe 3, Spalte 3:
    1  2  3  4  5 
 1  .  .  .  .  . 
 2  .  .  .  .  . 
 3  .  .  .  .  . 
 4  .  .  .  .  . 
 5  .  .  .  .  . 
Gewonnen: True


Die fünf LEDs in Form eines Kreuzes sind genau die Felder, die der Druck auf die mittlere Taste umschaltet. Ein zweiter Druck macht alles wieder rückgängig, deshalb ist das Brett jetzt leer.

## Schritt 4: Das Spiel mit Eingaben

Jetzt das eigentliche Spiel. Die Funktion `spiele(brett, regel)` zeigt das Brett, liest in einer Schleife die Züge (Reihe und Spalte, ab 1) und führt sie aus, bis alle LEDs aus sind.
Die Eingabe wird geprüft: Zwei Zahlen (sonst `ValueError` beim Umwandeln), und die Taste muss im Raster liegen. `q` bricht das Spiel ab.

`spiel_starten()` fragt zuerst nach der Größe, der Variante und den Positionen der anfangs eingeschalteten LEDs, dann beginnt das Spiel.

Die Zelle unten definiert nur die Funktionen. Starte das Spiel danach mit `spiel_starten()` in einer neuen Zelle. Wir führen es hier nicht aus, weil es auf Eingaben von dir wartet.
Ein Beispielablauf steht weiter unten.

In [ ]:
def lies_zahlen(text):
    zahlen = []
    for wort in text.split():
        zahlen.append(int(wort))
    return zahlen


def spiele(brett, regel):
    zeige(brett)
    while not gewonnen(brett):
        eingabe = input('Taste (Reihe Spalte), q zum Aufgeben: ')
        if eingabe.strip() == 'q':
            print('Aufgegeben.')
            return
        try:
            z, s = lies_zahlen(eingabe)
        except ValueError:
            print('Bitte genau zwei Zahlen eingeben, zum Beispiel: 3 3')
            continue
        if not (1 <= z <= len(brett) and 1 <= s <= len(brett[0])):
            print('Diese Taste gibt es nicht.')
            continue
        druecke(brett, z - 1, s - 1, regel)
        zeige(brett)
    print('Geschafft! Alle LEDs sind aus.')


def spiel_starten():
    reihen = int(input(f'Anzahl der Reihen (1 bis {MAX_GROESSE}): '))
    spalten = int(input(f'Anzahl der Spalten (1 bis {MAX_GROESSE}): '))
    if not (1 <= reihen <= MAX_GROESSE and 1 <= spalten <= MAX_GROESSE):
        print('Diese Größe ist nicht erlaubt.')
        return
    wahl = input('Variante: 1 = Nachbarn (Lights Out), 2 = links oben: ')
    regel = 'nachbarn' if wahl == '1' else 'obenlinks'
    zahlen = lies_zahlen(input('Welche LEDs sind am Anfang an? (Reihe Spalte Reihe Spalte ...): '))
    an = []
    for i in range(0, len(zahlen) - 1, 2):
        an.append((zahlen[i], zahlen[i + 1]))
    spiele(neues_brett(reihen, spalten, an), regel)

**Beispielablauf:** 3 Reihen, 4 Spalten, Variante Lights Out. Anfangs sind die LEDs (1, 1), (1, 2) und (2, 1) an. Zuerst gibt der Spieler zwei ungültige Eingaben ein (`x` und `9 9`),
dann die Taste `1 1`:

```
Anzahl der Reihen (1 bis 20): 3
Anzahl der Spalten (1 bis 20): 4
Variante: 1 = Nachbarn (Lights Out), 2 = links oben: 1
Welche LEDs sind am Anfang an? (Reihe Spalte Reihe Spalte ...): 1 1 2 1 1 2
    1  2  3  4 
 1  #  #  .  . 
 2  #  .  .  . 
 3  .  .  .  . 
Taste (Reihe Spalte), q zum Aufgeben: x
Bitte genau zwei Zahlen eingeben, zum Beispiel: 3 3
Taste (Reihe Spalte), q zum Aufgeben: 9 9
Diese Taste gibt es nicht.
Taste (Reihe Spalte), q zum Aufgeben: 1 1
    1  2  3  4 
 1  .  .  .  . 
 2  .  .  .  . 
 3  .  .  .  . 
Geschafft! Alle LEDs sind aus.
```

Die ungültigen Eingaben werden abgefangen, das Spiel läuft weiter. Der Druck auf die Taste (1, 1) schaltet das Feld (1, 1) und seine Nachbarn (1, 2) und (2, 1) um. Weil genau diese drei Felder an
waren, sind danach alle aus und das Spiel ist gewonnen.

## Schritt 5: Andere Größen und die zweite Variante

Weil alles mit `len(brett)` und `len(brett[0])` arbeitet, funktionieren auch andere Größen ohne Änderung. Hier ein 3 × 4 Brett mit der Regel `nachbarn` und ein 6 × 6 Brett mit der Regel `obenlinks`.
Bei `obenlinks` schaltet die Taste (4, 3) alle Felder in den Reihen 1 bis 4 und den Spalten 1 bis 3 um.

In [ ]:
brett = neues_brett(3, 4, [(1, 1), (1, 2), (2, 1)])
zeige(brett)
druecke(brett, 0, 0, 'nachbarn')
print('Nach dem Druck auf (1, 1):')
zeige(brett)
print('Gewonnen:', gewonnen(brett))
print()

brett = neues_brett(6, 6, [(1, 1), (2, 3), (4, 3), (4, 5)])
print('Variante links oben, Start:')
zeige(brett)
druecke(brett, 3, 2, 'obenlinks')           # Taste in Reihe 4, Spalte 3
print('Nach dem Druck auf Reihe 4, Spalte 3:')
zeige(brett)

    1  2  3  4 
 1  #  #  .  . 
 2  #  .  .  . 
 3  .  .  .  . 
Nach dem Druck auf (1, 1):
    1  2  3  4 
 1  .  .  .  . 
 2  .  .  .  . 
 3  .  .  .  . 
Gewonnen: True

Variante links oben, Start:
    1  2  3  4  5  6 
 1  #  .  .  .  .  . 
 2  .  .  #  .  .  . 
 3  .  .  .  .  .  . 
 4  .  .  #  .  #  . 
 5  .  .  .  .  .  . 
 6  .  .  .  .  .  . 
Nach dem Druck auf Reihe 4, Spalte 3:
    1  2  3  4  5  6 
 1  .  #  #  .  .  . 
 2  #  #  .  .  .  . 
 3  #  #  #  .  .  . 
 4  #  #  .  .  #  . 
 5  .  .  .  .  .  . 
 6  .  .  .  .  .  . 


Die Felder in Reihe 1 bis 4 und Spalte 1 bis 3 wurden umgeschaltet: Aus `#` wurde `.` und umgekehrt (auch die Felder, die vorher aus waren, sind jetzt an).

## Für Neugierige: Das Spiel automatisch lösen

**Die Variante „links oben“.** Hier gibt es eine einfache Lösung. Wir gehen alle Felder von **rechts unten** nach links oben durch (Reihe für Reihe von unten, in jeder Reihe von rechts nach links).
Ist die LED am aktuellen Feld an, drücken wir die Taste dort. Der Druck schaltet nur Felder links davon und oberhalb um, also nur solche, die wir noch nicht angeschaut haben. Was schon fertig ist, bleibt aus.
Diese Variante ist deshalb **immer lösbar**, und die Lösung ist eindeutig.

**Lights Out (Nachbarn).** Hier hilft ein Trick, das „Lichter jagen“: Man legt fest, welche Tasten der ersten Reihe gedrückt werden (bei 5 Spalten gibt es nur `2⁵ = 32` Möglichkeiten). Danach
ist alles andere **erzwungen**: Brennt in der Reihe darüber ein Licht, kann es nur durch die Taste direkt darunter ausgeschaltet werden. Man arbeitet sich so Reihe für Reihe nach unten.
Sind am Ende alle Lichter aus, ist das eine Lösung. Wir probieren alle 32 Möglichkeiten und nehmen die Lösung mit den wenigsten Zügen.
Jede Taste wird höchstens einmal gedrückt, denn zweimal drücken hebt sich auf, und die Reihenfolge ist egal.

In [ ]:
def kopie(brett):
    neu = []
    for zeile in brett:
        neu.append(list(zeile))
    return neu


def loese_obenlinks(brett):
    brett = kopie(brett)
    zuege = []
    for z in range(len(brett) - 1, -1, -1):
        for s in range(len(brett[0]) - 1, -1, -1):
            if brett[z][s] == 1:
                druecke(brett, z, s, 'obenlinks')
                zuege.append((z + 1, s + 1))            # ab 1 gezählt
    return zuege


def loese_nachbarn(brett):
    reihen = len(brett)
    spalten = len(brett[0])
    beste = None
    for maske in range(2 ** spalten):
        probe = kopie(brett)
        zuege = []
        for s in range(spalten):
            if maske // (2 ** s) % 2 == 1:              # Bit s der Zahl `maske`: erste Reihe, Taste s drücken
                druecke(probe, 0, s, 'nachbarn')
                zuege.append((1, s + 1))
        for z in range(1, reihen):
            for s in range(spalten):
                if probe[z - 1][s] == 1:                # Licht in der Reihe darüber: Taste darunter drücken
                    druecke(probe, z, s, 'nachbarn')
                    zuege.append((z + 1, s + 1))
        if gewonnen(probe) and (beste is None or len(zuege) < len(beste)):
            beste = zuege
    return beste


import random

# Eine Stellung der Variante links oben: wir mischen mit zufälligen Zügen
random.seed(2)
brett = neues_brett(6, 6, [])
for i in range(7):
    druecke(brett, random.randrange(6), random.randrange(6), 'obenlinks')
zeige(brett)
loesung = loese_obenlinks(brett)
print('Lösung (Reihe, Spalte):', loesung)
for z, s in loesung:
    druecke(brett, z - 1, s - 1, 'obenlinks')
print('Danach gewonnen:', gewonnen(brett))

    1  2  3  4  5  6 
 1  #  .  .  .  .  # 
 2  .  .  .  #  #  # 
 3  .  .  .  #  #  . 
 4  #  #  #  .  .  . 
 5  #  #  #  .  .  . 
 6  #  #  #  .  .  . 
Lösung (Reihe, Spalte): [(6, 3), (3, 5), (2, 6), (2, 5), (1, 5), (1, 3), (1, 1)]
Danach gewonnen: True


In [ ]:
# Eine Lights-Out-Stellung auf 5 x 5: wir mischen mit 8 zufälligen Zügen (dann ist sie lösbar)
random.seed(4)
brett = neues_brett(5, 5, [])
for i in range(8):
    druecke(brett, random.randrange(5), random.randrange(5), 'nachbarn')
zeige(brett)
loesung = loese_nachbarn(brett)
print('Lösung (Reihe, Spalte):', loesung)
for z, s in loesung:
    druecke(brett, z - 1, s - 1, 'nachbarn')
print('Danach gewonnen:', gewonnen(brett))

    1  2  3  4  5 
 1  .  .  .  .  . 
 2  #  .  #  #  . 
 3  .  #  #  .  . 
 4  #  #  .  .  # 
 5  .  .  #  .  # 
Lösung (Reihe, Spalte): [(1, 1), (1, 2), (2, 3), (4, 2), (5, 3), (5, 5)]
Danach gewonnen: True


Nicht jede Anfangsstellung bei Lights Out ist lösbar! Wenn man eine Stellung nicht durch Drücken von einem leeren Brett aus erhalten hat, kann es sein, dass die Suche keine Lösung findet
(`loese_nachbarn` gibt dann `None` zurück). Auf dem 5 × 5-Brett ist zum Beispiel ein einzelnes Licht in der Ecke **nicht** lösbar. Wir probieren alle 25 Stellungen mit genau einem Licht aus:
`L` steht für „lösbar“, `.` für „nicht lösbar“.

In [ ]:
for z in range(1, 6):
    text = ''
    for s in range(1, 6):
        einzeln = neues_brett(5, 5, [(z, s)])
        text = text + (' L' if loese_nachbarn(einzeln) is not None else ' .')
    print(text)

 . . . . .
 . L . L .
 . . L . .
 . L . L .
 . . . . .


Nur 5 der 25 Stellungen mit einem einzelnen Licht lassen sich lösen (die Mitte und die vier Felder daneben diagonal). Insgesamt sind bei 5 × 5 nur ein Viertel aller Stellungen lösbar.
Für ein Spiel auf dem LAMA sollte man die Anfangsstellung deshalb durch Drücken von einem leeren Brett aus erzeugen (zufällig mischen), dann ist sie immer lösbar.